In [0]:
df = spark.range(1, 11)
display(df)

In [0]:
files = dbutils.fs.ls("/Volumes/workspace/bronze/scada_raw_files/")

display(files)

In [0]:
df_scada = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/bronze/scada_raw_files/*.csv")
)

display(df_scada.limit(10))

In [0]:
df_scada = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/bronze/scada_raw_files/*.csv")
)

display(df_scada.limit(10))

In [0]:
df_scada.printSchema()

In [0]:
df_scada.select("value").distinct().show(20, truncate=False)

In [0]:
from pyspark.sql.functions import col

df_scada.filter(
    col("value").rlike(r"^[0-9]+\.[0-9]+\.[0-9]+$")
).select("id", "value", "unit").show(20, truncate=False)

In [0]:
df_scada.filter(
    col("value").rlike(r"^[0-9]+\.[0-9]+\.[0-9]+$")
).count()

In [0]:
df_scada.count()

In [0]:
df_scada.filter(
    col("value").rlike(r"^[0-9]+\.[0-9]+\.[0-9]+$")
).groupBy("id").count().orderBy("id").show(50)

In [0]:
df_scada.filter(
    col("id") == 64
).select(
    "id", "value", "unit", "timestamp"
).show(30, truncate=False)

In [0]:
spark.read.text(
    "/Volumes/workspace/bronze/scada_raw_files/"
).filter(
    col("value").contains("1.410.758")
).show(10, truncate=False)

In [0]:
spark.read.text(
    "/Volumes/workspace/bronze/scada_raw_files/"
).show(5, truncate=False)

In [0]:
bronze_df = df_scada.select(
    "id",
    "value",
    "unit",
    "timestamp"
)

bronze_df.printSchema()
bronze_df.show(10, truncate=False)

In [0]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.scada_sensor_raw")

In [0]:
spark.table("workspace.bronze.scada_sensor_raw").show(10, truncate=False)

In [0]:
spark.table("workspace.bronze.scada_sensor_raw").count()

In [0]:
from pyspark.sql.functions import col, sum

bronze_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in bronze_df.columns
]).show()

In [0]:
bronze_df.filter(
    col("id").isNull() &
    col("value").isNull() &
    col("unit").isNull() &
    col("timestamp").isNull()
).count()

In [0]:
silver_df = bronze_df.dropna(
    how="all",
    subset=["id", "value", "unit", "timestamp"]
)

silver_df.count()

In [0]:
silver_df.select("value").distinct().show(30, truncate=False)

In [0]:
from pyspark.sql.functions import col

silver_df.filter(
    col("value").rlike(r"^[0-9]+\.[0-9]+\.[0-9]+$")
).count()

In [0]:
silver_df.filter(
    col("value").rlike(r"^[0-9]+\.[0-9]+\.[0-9]+$")
).groupBy("id").count().orderBy("id").show(100)

In [0]:
silver_df.filter(
    col("id") == 64
).select(
    "id", "value", "unit", "timestamp"
).orderBy("timestamp").show(30, truncate=False)

In [0]:
from pyspark.sql.functions import regexp_count, lit

silver_df.groupBy(
    regexp_count(col("value"), lit("[.]")).alias("dot_count")
).count().orderBy("dot_count").show()

In [0]:
silver_df.filter(
    ~col("value").contains(".")
).select(
    "id", "value", "unit"
).show(30, truncate=False)

In [0]:
from pyspark.sql.functions import regexp_replace

silver_df.filter(
    col("value").rlike(r"^[0-9]+\.[0-9]+\.[0-9]+$")
).select(
    "value",
    regexp_replace(
        col("value"),
        r"^([0-9]+)\.([0-9]+)\.([0-9]+)$",
        "$1$2.$3"
    ).alias("cleaned_value")
).show(20, truncate=False)

In [0]:
from pyspark.sql.functions import when, col, regexp_replace

silver_df = silver_df.withColumn(
    "value",
    when(
        col("value").rlike(r"^[0-9]+\.[0-9]+\.[0-9]+$"),
        regexp_replace(
            col("value"),
            r"^([0-9]+)\.([0-9]+)\.([0-9]+)$",
            "$1$2.$3"
        )
    )
    .otherwise(col("value"))
    .cast("double")
)

silver_df.printSchema()

In [0]:
silver_df.filter(
    col("value").isNull()
).count()

In [0]:
total_rows = silver_df.count()
unique_rows = silver_df.dropDuplicates().count()

print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", total_rows - unique_rows)

In [0]:
silver_df.groupBy("unit").count().orderBy("unit").show(50, truncate=False)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.scada_sensor_clean")

In [0]:
from pyspark.sql.functions import count, avg, min, max

gold_sensor_summary = silver_df.groupBy("id", "unit").agg(
    count("*").alias("reading_count"),
    avg("value").alias("avg_value"),
    min("value").alias("min_value"),
    max("value").alias("max_value")
)

gold_sensor_summary.orderBy("id").show(50, truncate=False)

In [0]:
gold_sensor_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.sensor_summary")

In [0]:
gold_df = spark.table("workspace.gold.sensor_summary")

print("Gold rows:", gold_df.count())
gold_df.orderBy("id").show(20, truncate=False)

In [0]:
base_path = "/Volumes/workspace/bronze/scada_raw_files/autoloader_demo"

incoming_path = f"{base_path}/incoming"
checkpoint_path = f"{base_path}/checkpoint"
schema_path = f"{base_path}/schema"

dbutils.fs.mkdirs(incoming_path)
dbutils.fs.mkdirs(checkpoint_path)
dbutils.fs.mkdirs(schema_path)

print("Incoming:", incoming_path)
print("Checkpoint:", checkpoint_path)
print("Schema:", schema_path)

In [0]:
bronze_df = spark.table("workspace.bronze.scada_sensor_raw")

incoming_path = "/Volumes/workspace/bronze/scada_raw_files/autoloader_demo/incoming"

print("Bronze rows:", bronze_df.count())

In [0]:
batch1_df = bronze_df.limit(100000)

(
    batch1_df
    .coalesce(1)
    .write
    .mode("append")
    .option("header", "true")
    .option("delimiter", ";")
    .csv(incoming_path)
)

print("Batch 1 created")

In [0]:
schema_path = "/Volumes/workspace/bronze/scada_raw_files/autoloader_demo/schema"

autoloader_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .option("delimiter", ";")
    .load(incoming_path)
)

autoloader_df.printSchema()

In [0]:
checkpoint_path = "/Volumes/workspace/bronze/scada_raw_files/autoloader_demo/checkpoint"

query = (
    autoloader_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.bronze.scada_sensor_stream_raw")
)

query.awaitTermination()

print("Auto Loader Batch 1 completed")

In [0]:
stream_bronze_df = spark.table(
    "workspace.bronze.scada_sensor_stream_raw"
)

print("Streaming Bronze rows:", stream_bronze_df.count())

display(stream_bronze_df.limit(10))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.orderBy("timestamp", "id", "value")

numbered_df = bronze_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

batch2_df = (
    numbered_df
    .filter((col("row_num") > 100000) & (col("row_num") <= 200000))
    .drop("row_num")
)

print("Batch 2 rows:", batch2_df.count())

In [0]:
(
    batch2_df
    .coalesce(1)
    .write
    .mode("append")
    .option("header", "true")
    .option("delimiter", ";")
    .csv(incoming_path)
)

print("Batch 2 file arrived")

In [0]:
query2 = (
    autoloader_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.bronze.scada_sensor_stream_raw")
)

query2.awaitTermination()

print("Auto Loader Batch 2 completed")

In [0]:
stream_bronze_df = spark.table(
    "workspace.bronze.scada_sensor_stream_raw"
)

print("Streaming Bronze rows:", stream_bronze_df.count())

In [0]:
incoming_files = [
    f.name
    for f in dbutils.fs.ls(incoming_path)
    if f.name.endswith(".csv")
]

print("CSV files in incoming:", len(incoming_files))

for f in incoming_files:
    print(f)

In [0]:
debug_df = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .csv(incoming_path)
    .withColumn("source_file", col("_metadata.file_path"))
)

debug_df.groupBy("source_file").count().show(truncate=False)

In [0]:
from pyspark.sql.functions import regexp_extract

debug_df.groupBy("source_file").count().select(
    regexp_extract("source_file", r"([^/]+)$", 1).alias("file_name"),
    "count"
).show(truncate=False)

In [0]:
spark.sql("""
DROP TABLE IF EXISTS workspace.bronze.scada_sensor_stream_raw
""")

dbutils.fs.rm(
    "/Volumes/workspace/bronze/scada_raw_files/autoloader_demo",
    True
)

print("Auto Loader demo reset completed")

In [0]:
base_path = "/Volumes/workspace/bronze/scada_raw_files/autoloader_demo"

incoming_path = f"{base_path}/incoming"
checkpoint_path = f"{base_path}/checkpoint"
schema_path = f"{base_path}/schema"

dbutils.fs.mkdirs(incoming_path)
dbutils.fs.mkdirs(checkpoint_path)
dbutils.fs.mkdirs(schema_path)

print("Auto Loader folders recreated")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

bronze_df = spark.table("workspace.bronze.scada_sensor_raw")

window_spec = Window.orderBy("timestamp", "id", "value")

numbered_df = bronze_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

batch1_df = (
    numbered_df
    .filter(col("row_num") <= 100000)
    .drop("row_num")
)

print("Batch 1 rows:", batch1_df.count())

In [0]:
temp_path = f"{base_path}/batch1_temp"
batch1_file = f"{incoming_path}/batch1.csv"

# Clean temporary folder if it exists
dbutils.fs.rm(temp_path, True)

# Write Batch 1
(
    batch1_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("delimiter", ";")
    .csv(temp_path)
)

# Find the generated CSV
part_file = [
    f.path
    for f in dbutils.fs.ls(temp_path)
    if f.name.endswith(".csv")
][0]

# Rename it to a fixed filename
dbutils.fs.mv(part_file, batch1_file)

# Remove temporary folder
dbutils.fs.rm(temp_path, True)

print("Batch 1 file created:", batch1_file)

In [0]:
files = [
    f.name
    for f in dbutils.fs.ls(incoming_path)
    if f.name.endswith(".csv")
]

print("Incoming CSV files:", len(files))
print(files)

In [0]:
schema_path = "/Volumes/workspace/bronze/scada_raw_files/autoloader_demo/schema"
checkpoint_path = "/Volumes/workspace/bronze/scada_raw_files/autoloader_demo/checkpoint"

autoloader_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .option("delimiter", ";")
    .load(incoming_path)
)

query = (
    autoloader_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.bronze.scada_sensor_stream_raw")
)

query.awaitTermination()

print("Streaming Bronze Batch 1 completed")

In [0]:
stream_bronze_df = spark.table(
    "workspace.bronze.scada_sensor_stream_raw"
)

print("Streaming Bronze rows:", stream_bronze_df.count())

In [0]:
from pyspark.sql.functions import col

batch2_df = (
    numbered_df
    .filter((col("row_num") > 100000) & (col("row_num") <= 200000))
    .drop("row_num")
)

print("Batch 2 rows:", batch2_df.count())

In [0]:
temp_path = f"{base_path}/batch2_temp"
batch2_file = f"{incoming_path}/batch2.csv"

dbutils.fs.rm(temp_path, True)

(
    batch2_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("delimiter", ";")
    .csv(temp_path)
)

part_file = [
    f.path
    for f in dbutils.fs.ls(temp_path)
    if f.name.endswith(".csv")
][0]

dbutils.fs.mv(part_file, batch2_file)
dbutils.fs.rm(temp_path, True)

print("Batch 2 file created:", batch2_file)

In [0]:
query2 = (
    autoloader_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.bronze.scada_sensor_stream_raw")
)

query2.awaitTermination()

print("Streaming Bronze Batch 2 completed")

In [0]:
stream_bronze_df = spark.table(
    "workspace.bronze.scada_sensor_stream_raw"
)

print("Streaming Bronze rows:", stream_bronze_df.count())

In [0]:
stream_bronze_df = spark.table(
    "workspace.bronze.scada_sensor_stream_raw"
)

stream_bronze_df.printSchema()

In [0]:
stream_bronze_df = spark.table(
    "workspace.bronze.scada_sensor_stream_raw"
)

stream_bronze_df.printSchema()

In [0]:
from pyspark.sql.functions import col, when, regexp_replace

stream_source_df = spark.readStream.table(
    "workspace.bronze.scada_sensor_stream_raw"
)

stream_silver_df = (
    stream_source_df
    .dropna(
        how="all",
        subset=["id", "value", "unit", "timestamp"]
    )
    .withColumn("id", col("id").cast("int"))
    .withColumn(
        "value",
        when(
            col("value").rlike(r"^[0-9]+\.[0-9]+\.[0-9]+$"),
            regexp_replace(
                col("value"),
                r"^([0-9]+)\.([0-9]+)\.([0-9]+)$",
                "$1$2.$3"
            )
        )
        .otherwise(col("value"))
        .cast("double")
    )
    .withColumn(
        "timestamp",
        col("timestamp").cast("timestamp")
    )
    .select(
        "id",
        "value",
        "unit",
        "timestamp"
    )
)

stream_silver_df.printSchema()

In [0]:
stream_silver_checkpoint = (
    "/Volumes/workspace/bronze/scada_raw_files/"
    "autoloader_demo/silver_checkpoint"
)

silver_query = (
    stream_silver_df.writeStream
    .format("delta")
    .option("checkpointLocation", stream_silver_checkpoint)
    .trigger(availableNow=True)
    .toTable("workspace.silver.scada_sensor_stream_clean")
)

silver_query.awaitTermination()

print("Streaming Silver completed")

In [0]:
stream_silver_saved = spark.table(
    "workspace.silver.scada_sensor_stream_clean"
)

print("Streaming Silver rows:", stream_silver_saved.count())

stream_silver_saved.printSchema()

In [0]:
from pyspark.sql.functions import count, avg, min, max

stream_gold_df = (
    spark.readStream
    .table("workspace.silver.scada_sensor_stream_clean")
    .groupBy("id", "unit")
    .agg(
        count("*").alias("reading_count"),
        avg("value").alias("avg_value"),
        min("value").alias("min_value"),
        max("value").alias("max_value")
    )
)

stream_gold_df.printSchema()

In [0]:
stream_gold_checkpoint = (
    "/Volumes/workspace/bronze/scada_raw_files/"
    "autoloader_demo/gold_checkpoint"
)

gold_query = (
    stream_gold_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", stream_gold_checkpoint)
    .trigger(availableNow=True)
    .toTable("workspace.gold.sensor_stream_summary")
)

gold_query.awaitTermination()

print("Streaming Gold completed")

In [0]:
stream_gold_saved = spark.table(
    "workspace.gold.sensor_stream_summary"
)

print("Streaming Gold rows:", stream_gold_saved.count())

stream_gold_saved.orderBy("id").show(20, truncate=False)

In [0]:
from pyspark.sql.functions import sum

stream_gold_saved.agg(
    sum("reading_count").alias("total_readings")
).show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

bronze_df = spark.table(
    "workspace.bronze.scada_sensor_raw"
)

window_spec = Window.orderBy(
    "timestamp", "id", "value"
)

numbered_df = bronze_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

batch3_df = (
    numbered_df
    .filter(
        (col("row_num") > 200000) &
        (col("row_num") <= 300000)
    )
    .drop("row_num")
)

print("Batch 3 rows:", batch3_df.count())

In [0]:
base_path = "/Volumes/workspace/bronze/scada_raw_files/autoloader_demo"
incoming_path = f"{base_path}/incoming"

temp_path = f"{base_path}/batch3_temp"
batch3_file = f"{incoming_path}/batch3.csv"

dbutils.fs.rm(temp_path, True)

(
    batch3_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("delimiter", ";")
    .csv(temp_path)
)

part_file = [
    f.path
    for f in dbutils.fs.ls(temp_path)
    if f.name.endswith(".csv")
][0]

dbutils.fs.mv(part_file, batch3_file)
dbutils.fs.rm(temp_path, True)

print("Batch 3 file arrived:", batch3_file)